# Point ROI Orthorectification — Small Region Around Lat/Lon

**Objective**: Extract and orthorectify a small region (ROI) centered on a specific geographic point (lat/lon).

## Preamble

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(f"GRDL {required}+ required.")

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.geolocation.chip import ChipGeolocation
from grdl.image_processing.ortho import orthorectify, UTMGrid

## Configuration

In [ ]:
SICD_PATH = Path("/path/to/your/sicd_file.nitf")
ROI_LAT = 34.05
ROI_LON = -118.25
ROI_RADIUS_M = 250.0  # Meters around point

assert SICD_PATH.exists(), f"File not found: {SICD_PATH}"
print(f"✓ SICD: {SICD_PATH.name}")
print(f"  ROI: ({ROI_LAT}, {ROI_LON}), radius={ROI_RADIUS_M}m")

## Step 1: Find Pixel Coordinates of ROI Center

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    meta = reader.metadata
    geo = create_geolocation(reader)
    
    # Convert lat/lon to row/col
    roi_row, roi_col = geo.latlon_to_image(ROI_LAT, ROI_LON)[0:2]
    
    print(f"ROI center pixel: ({roi_row:.1f}, {roi_col:.1f})")
    
    # Estimate chip size from radius
    pixel_spacing_m = meta.grid.row.ss
    chip_size = int(2 * ROI_RADIUS_M / pixel_spacing_m)
    
    r_start = max(0, int(roi_row - chip_size//2))
    c_start = max(0, int(roi_col - chip_size//2))
    r_end = min(meta.image_data.num_rows, r_start + chip_size)
    c_end = min(meta.image_data.num_cols, c_start + chip_size)
    
    roi_chip = reader.read(row_start=r_start, row_end=r_end, col_start=c_start, col_end=c_end)

print(f"Extracted ROI chip: {roi_chip.shape}")

## Step 2: Orthorectify ROI to UTM

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    geo_full = create_geolocation(reader)
    geo = ChipGeolocation(geo_full, row_offset=r_start, col_offset=c_start)

output_grid = UTMGrid.from_center(
    center_lat=ROI_LAT,
    center_lon=ROI_LON,
    width_m=2*ROI_RADIUS_M,
    height_m=2*ROI_RADIUS_M,
    pixel_size_m=0.5,
)

magnitude = np.abs(roi_chip)
ortho_result = orthorectify(magnitude, geo, output_grid, interpolation='bilinear')
ortho_roi = ortho_result.data

print(f"✓ Orthorectified ROI: {ortho_roi.shape}")

## Step 3: Visualization

In [ ]:
viewer = napari.Viewer(title="Point ROI Ortho")

mag_db = 20*np.log10(magnitude + 1e-12)
ortho_db = 20*np.log10(ortho_roi + 1e-12)

viewer.add_image(mag_db, name="ROI Slant Range", colormap="gray")
viewer.add_image(ortho_db, name="ROI UTM Ortho", colormap="gray", visible=True)

print("✓ Napari viewer launched")
print(f"  ROI centered at ({ROI_LAT}, {ROI_LON})")

## Summary

✅ Point-centered ROI extraction
✅ Geographic point → pixel → chip → ortho pipeline
✅ Small-region high-detail orthorectification